# Practical Session: Tokenizers
---
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/paulnovello/Advanced-AI/blob/main/PP4%3A%20Tokenizer/tokenizer.ipynb)

## ⚙️ Setup

We need four libraries:
- **`transformers`** — HuggingFace's library for loading pre-trained tokenizers and models (GPT-2, BERT, Mistral…)
- **`tokenizers`** — the fast, low-level Rust-backed library for training new tokenizers from scratch
- **`datasets`** — for loading standard NLP datasets (Wikitext, IMDB…)
- **`torch`** — PyTorch, the deep learning framework we'll use to run model forward passes


In [ ]:
!pip install transformers tokenizers datasets torch -q

In [ ]:
from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification,
    GPT2LMHeadModel, GPT2Config,
    DataCollatorWithPadding, DataCollatorForLanguageModeling,
)
from tokenizers import Tokenizer, models, pre_tokenizers, trainers, decoders
from datasets import load_dataset
from torch.utils.data import DataLoader
import torch
import random
import textwrap
import unicodedata

### Quick glossary: symbols you'll encounter in this notebook

Throughout this notebook you'll see special symbols in tokenizer outputs. Here's what they mean:

| Symbol | Name | Meaning |
|--------|------|---------|
| `Ġ` | Leading space marker | Indicates that this token was preceded by a space in the original text. Used by **ByteLevel** tokenizers (GPT-2, LLaMA). For example, `Ġthe` means `" the"`. |
| `##` | WordPiece continuation | Used by **BERT**. A token starting with `##` is a continuation of the previous token. For example, `["token", "##ization"]` → `"tokenization"`. |
| `▁` | SentencePiece space | Used by **SentencePiece** models (T5, LLaMA). Similar to `Ġ` — marks word boundaries. `▁The` means `" The"`. |
| `[UNK]` | Unknown token | A fallback for characters not in the vocabulary. ByteLevel tokenizers avoid this entirely since every byte is representable. |
| `[PAD]` | Padding token | Fills shorter sequences in a batch so all sequences have equal length. The model learns to **ignore** these via the attention mask. |
| `[BOS]` | Beginning of sequence | Marks the start of a text. Not all models use this (GPT-2 doesn't, LLaMA does). |
| `[EOS]` / `<\|endoftext\|>` | End of sequence | Marks the end of a text. GPT-2 uses `<\|endoftext\|>`, BERT uses `[SEP]`. |
| `[CLS]` | Classification token | BERT-specific. Placed at the start of every input. Its final hidden state is used as a "sentence representation" for classification tasks. |
| `[SEP]` | Separator token | BERT-specific. Separates two sentences in a pair (e.g., question + passage for QA). Also marks the end of a sequence. |
| `[MASK]` | Mask token | BERT-specific. Used during pre-training to hide tokens the model must predict (masked language modeling). |
| `-100` | Ignore index | Not a token — it's PyTorch's convention for labels that should be **excluded from loss computation** (padding, continuation subtokens in NER). |

Don't worry about memorizing these now, we'll encounter each one in context as we go.

---
# Part 1 — Intuition & Tokenizer Comparison

Before writing a single tokenizer, let's understand what they do and why choices matter.

### The core question: What is a token?
A token is the *atomic unit of text* that a model sees. Models never read characters or words directly, they read token IDs, which are indices into a fixed vocabulary.

### 1.1 — Visual helper: color-coded tokenization

To compare tokenizers we need to **see** the splits they produce. The helper below prints each token in a different background color so boundaries are obvious at a glance.

In [ ]:
# A small helper to visualize tokens in color
COLORS = [
    "\033[41m", "\033[42m", "\033[43m", "\033[44m",
    "\033[45m", "\033[46m", "\033[103m", "\033[102m",
]
RESET = "\033[0m"

def show_tokens(text, tokenizer, tokenizer_name=""):
    """Print a sentence with each token highlighted in alternating colors."""
    tokens = tokenizer.tokenize(text)
    print(f"\n{'─'*60}")
    print(f"  Tokenizer: {tokenizer_name or tokenizer.__class__.__name__}")
    print(f"  Vocab size: {tokenizer.vocab_size:,}")
    print(f"  Token count: {len(tokens)}")
    print(f"{'─'*60}")
    colored = ""
    for i, tok in enumerate(tokens):
        color = COLORS[i % len(COLORS)]
        colored += f"{color} {repr(tok)} {RESET}"
    print(colored)
    print()

Now let's load three widely-used tokenizers and run them on the same inputs. Notice how each one splits text differently: different vocabulary, different granularity.

Each tokenizer was trained on a different corpus with a different algorithm:
- **GPT-2** uses Byte-Pair Encoding (BPE) trained on WebText (internet text)
- **BERT** uses WordPiece trained on Wikipedia + BookCorpus
- **Mistral** uses SentencePiece/BPE trained on a large multilingual web corpus


### 1.2 — Same sentence, different tokenizers

We'll compare **GPT-2** (BPE), **BERT** (WordPiece), and **LLaMA** (SentencePiece/BPE).  
Watch how vocabulary choices lead to very different splits.

In [ ]:
gpt2_tok    = AutoTokenizer.from_pretrained("gpt2")
bert_tok    = AutoTokenizer.from_pretrained("bert-base-uncased")
mistral_tok = AutoTokenizer.from_pretrained("mistralai/Mistral-7B-v0.1")  # SentencePiece/BPE, fully open

In [ ]:
sentence = "Tokenization is a crucial preprocessing step in NLP pipelines."

for tok, name in [(gpt2_tok, "GPT-2 (BPE)"), (bert_tok, "BERT (WordPiece)"), (mistral_tok, "Mistral (BPE/SentencePiece)")]:
    show_tokens(sentence, tok, name)

The cell above shows the same sentence through three lenses. Look at the **token count**. Fewer tokens means the tokenizer is more efficient for this kind of text. Any idea why?

Now let's try inputs that stress-test tokenizers: numbers, rare words, other languages, code, and emojis. These are the cases where tokenizer quality really matters.

In [ ]:
cases = [
    ("Numbers",    "The year 2024 had 365 days and 8760 hours."),
    ("Rare word",  "The antidisestablishmentarianism debate resurfaced."),
    ("Non-English","Le chat mange une souris dans la cuisine."),
    ("Code",       "for i in range(len(tokens)): print(tokens[i])"),
    ("Emojis",     "I ❤️ NLP 🤗 it's amazing!"),
]

for label, text in cases:
    print(f"\n{'='*60}")
    print(f"  CASE: {label}")
    print(f"  Text: {text}")
    show_tokens(text, gpt2_tok, "GPT-2")
    show_tokens(text, bert_tok, "BERT")
    show_tokens(text, mistral_tok, "Mistral")

**What to look for in the results above:**
- **Numbers** get split into individual digits by some tokenizers —> this makes arithmetic harder for models.
- **Rare words** are broken into many small pieces (high fragmentation = high fertility).
- **Non-English text** is handled poorly by tokenizers trained mostly on English data.
- **Code** tokens like `range`, `len`, `print` may or may not be single tokens depending on the tokenizer's training data.
- **Emojis** are usually encoded as multiple byte-level tokens.

**Notation you'll see in the outputs:**
- GPT-2 tokens starting with `Ġ` (e.g. `Ġthe`) — the `Ġ` represents a **leading space** (see glossary above).
- BERT tokens starting with `##` (e.g. `##ization`) — the `##` means this is a **continuation** of the previous token, not a new word.

These differences directly impact model performance: a model that sees a number as 7 separate tokens has a harder time reasoning about it than one that sees it as 1 token.

### 1.3 — Fertility Rate

> **Fertility** = average number of tokens per word.  
> A lower fertility means more efficient encoding — fewer tokens needed for the same text.  
> Typical well-trained tokenizers land between **1.1 and 1.5** on their target language.

In [ ]:
def fertility(text, tokenizer):
    # NOTE: text.split() approximates word count using whitespace splitting.
    # This is NOT linguistically accurate (e.g. "don't" counts as 1 word,
    # but is arguably 2). It's a rough proxy — good enough for comparison.
    words  = text.split()
    tokens = tokenizer.tokenize(text)
    return len(tokens) / len(words)

# Use a paragraph that mixes common & rare words
test_paragraph = """
The mitochondria is the powerhouse of the cell. Neural networks learn hierarchical
representations from data. The antidisestablishmentarianism movement in the 1800s
opposed the disestablishment of the Church of England. Python is a versatile
programming language used extensively in data science and machine learning.
"""

for tok, name in [(gpt2_tok, "GPT-2"), (bert_tok, "BERT"), (mistral_tok, "Mistral")]:
    f = fertility(test_paragraph, tok)
    print(f"{name:20s} → fertility = {f:.3f} tokens/word")

### ✏️ Exercise 1

1. Find a sentence where **BERT produces more tokens than GPT-2** on the same input. Why?
2. Try the sentence `"1000000 + 999999 = ?"`. How many tokens does each tokenizer use for the numbers? What risk does this create for arithmetic reasoning?
3. What does GPT-2's tokenizer do with leading whitespace? (Hint: inspect `gpt2_tok.tokenize(" hello")` vs `gpt2_tok.tokenize("hello")`.)

In [ ]:
# Q1: Find a sentence where BERT produces more tokens than GPT-2
# BERT is uncased and uses WordPiece which splits rare/compound words aggressively.
# GPT-2 uses BPE trained on a large web corpus, so it handles compound words better.


# Q2: Numeric tokenization

# Q3: Leading whitespace in GPT-2

---
# Part 2 — Train Your Own BPE Tokenizer

### How BPE works:
1. **Start** with a character-level vocabulary
2. **Count** the most frequent adjacent pair of symbols
3. **Merge** that pair into a new symbol, repeat until vocab size is reached

The result: common words become single tokens, rare words get split into frequent subwords.

### 2.1 — Load a Wikipedia subset

To train a tokenizer we need a **corpus**  (a large body of text). We'll use a subset of Wikitext-103 (English Wikipedia articles). The tokenizer will learn which character sequences appear frequently and should become single tokens.

We stream the dataset directly to a file to avoid loading everything into RAM, and filter out very short lines (headers, stubs) that add noise.

In [ ]:
# Stream Wikitext directly to file — avoids loading all texts into memory
corpus_path = "/tmp/wiki_corpus.txt"
N_ARTICLES = 50_000
MIN_LENGTH = 50  # skip empty/stub lines shorter than this many characters

print("Streaming Wikitext to corpus file...")
dataset = load_dataset(
    "wikitext",
    "wikitext-103-raw-v1",
    split="train",
    streaming=True
)

n_written = 0
total_chars = 0
with open(corpus_path, "w", encoding="utf-8") as f:
    for sample in dataset.take(N_ARTICLES):
        text = sample["text"].strip()
        if len(text) < MIN_LENGTH:
            continue  # skip empty lines, headers-only, stubs
        line = text.replace("\n", " ")
        f.write(line + "\n")
        n_written += 1
        total_chars += len(line)

print(f"Wrote {n_written:,} documents ({total_chars:,} chars) to {corpus_path}")

The corpus is saved to disk. Let's verify what a sample line looks like.

The quality of your training corpus directly impacts the tokenizer's vocabulary. If your corpus is mostly English Wikipedia, the tokenizer will be optimized for English prose and may perform poorly on code, other languages, or domain-specific text.


In [ ]:
# Quick preview of the corpus
with open(corpus_path, "r", encoding="utf-8") as f:
    first_line = f.readline()
print(f"Sample (first 300 chars):\n{first_line[:300]}")

### 2.2 — Train the BPE tokenizer

We'll train a **32,000-token vocabulary** — similar to early GPT models.  
Each step in the pipeline has a clear role:

In [ ]:
# ── 1. Model: BPE with an unknown token ─────────────────────────────────────
wiki_tokenizer = Tokenizer(models.BPE(unk_token="[UNK]"))

# ── 2. Pre-tokenizer: ByteLevel ─────────────────────────────────────────────
#    Maps all bytes to visible unicode characters using GPT-2's byte-to-unicode
#    mapping. The Ġ character you'll see in the vocab represents a leading space.
#    Unlike simple whitespace splitting, ByteLevel ensures every byte is
#    representable — this means the tokenizer can handle ANY input (no [UNK]).
wiki_tokenizer.pre_tokenizer = pre_tokenizers.ByteLevel(add_prefix_space=False)

# ── 3. Trainer: configure vocabulary size and special tokens ────────────────
trainer = trainers.BpeTrainer(
    vocab_size=32_000,
    min_frequency=2,          # pair must appear ≥2 times to be merged
    special_tokens=["[UNK]", "[PAD]", "[BOS]", "[EOS]"],
    show_progress=True,
)

# ── 4. Decoder: reassemble tokens back into text ────────────────────────────
wiki_tokenizer.decoder = decoders.ByteLevel()

# ── 5. Train! ────────────────────────────────────────────────────────────────
print("Training BPE tokenizer...")
wiki_tokenizer.train([corpus_path], trainer)
print(f"\nDone! Vocabulary size: {wiki_tokenizer.get_vocab_size():,}")

Good practice: always save and reload your tokenizer. This confirms the serialization works and gives you a portable `.json` file you can share or version-control.

In [ ]:
# Save and reload (shows the standard save/load workflow)
wiki_tokenizer.save("/tmp/wiki_tokenizer.json")
wiki_tokenizer = Tokenizer.from_file("/tmp/wiki_tokenizer.json")
print("Tokenizer saved and reloaded.")

The tokenizer is trained. Let's now inspect **what it learned**: which tokens ended up in the vocabulary, and how our tokenizer compares to GPT-2 on the same sentences.

### 2.3 — Inspect what was learned

When you look at the vocabulary, you'll notice many tokens start with `Ġ` (e.g., `Ġthe`, `Ġand`). This is the **ByteLevel** marker for a leading space — `Ġthe` actually represents `" the"` (with a space before it). This is how the tokenizer preserves word boundaries without needing an explicit whitespace token.

In [ ]:
# Look at the vocabulary — sorted by token ID
vocab = wiki_tokenizer.get_vocab()  # dict: token -> id
vocab_by_id = sorted(vocab.items(), key=lambda x: x[1])

print("First 30 tokens (character-level, assigned first):")
for token, idx in vocab_by_id[:30]:
    print(f"  [{idx:5d}] {repr(token)}")

print("\nLast 20 tokens (most frequent long merges):")
for token, idx in vocab_by_id[-20:]:
    print(f"  [{idx:5d}] {repr(token)}")

The **first tokens** in the vocabulary are single characters (the initial alphabet before any merges). The **last tokens** are the longest merged sequences — common English words and phrases that appeared frequently enough to earn their own token ID.

Now let's compare our tokenizer against GPT-2 on several sentence types:

In [ ]:
# Compare our tokenizer vs GPT-2 on the same sentences
def encode_and_show(text, label=""):
    output = wiki_tokenizer.encode(text)
    gpt2_tokens = gpt2_tok.tokenize(text)

    print(f"\n{'─'*60}")
    if label:
        print(f"  {label}")
    print(f"  Text: {text[:80]}")
    print(f"  Our tokenizer  ({len(output.tokens):3d} tokens): {output.tokens[:15]}")
    print(f"  GPT-2          ({len(gpt2_tokens):3d} tokens): {gpt2_tokens[:15]}")

test_sentences = [
    ("Common English",    "The history of the United States began with colonization by European settlers."),
    ("Science",           "Photosynthesis converts carbon dioxide and water into glucose using sunlight."),
    ("Numbers",           "The population of Earth reached 8 billion in November 2022."),
    ("Rare/technical",    "The mitochondrial DNA encodes 13 polypeptides of the oxidative phosphorylation."),
]

for label, sent in test_sentences:
    encode_and_show(sent, label)

Both tokenizers produce similar token counts on common English text (both were trained on English). Differences appear on technical or rare vocabulary, where training data coverage matters.

Let's quantify this with the **fertility metric** on a held-out paragraph:

### 2.4 — Fertility comparison: ours vs GPT-2

Now let's quantitatively compare our freshly trained tokenizer against GPT-2 on held-out text (text that was **not** in our training corpus). This tells us how well our tokenizer generalizes.

Since both tokenizers were trained on English text, we expect similar fertility on English prose. Differences reveal vocabulary coverage gaps.


In [ ]:
# Hold-out text (not in our training corpus)
eval_text = """
Wikipedia is a free, multilingual online encyclopedia written and maintained by a
community of volunteers through open collaboration and using a wiki-based editing system.
It is the largest and most-read reference work in history, and has consistently been one
of the ten most popular websites ranked by Similarweb.
"""

wiki_tokens  = wiki_tokenizer.encode(eval_text).tokens
gpt2_tokens  = gpt2_tok.tokenize(eval_text)
bert_tokens  = bert_tok.tokenize(eval_text)
words        = eval_text.split()

print(f"Words:               {len(words):4d}")
print(f"Our BPE tokens:      {len(wiki_tokens):4d}  (fertility = {len(wiki_tokens)/len(words):.3f})")
print(f"GPT-2 tokens:        {len(gpt2_tokens):4d}  (fertility = {len(gpt2_tokens)/len(words):.3f})")
print(f"BERT tokens:         {len(bert_tokens):4d}  (fertility = {len(bert_tokens)/len(words):.3f})")

Our tokenizer was trained on Wikipedia too, so it should perform similarly to GPT-2.
It is not exactly the case because:  
1. Corpus size difference. Our tokenizer was trained on ~50,000 Wikitext lines. GPT-2's tokenizer was trained on WebText — ~8 million web pages. More data means BPE sees far more co-occurrence patterns and learns better merges.

2. Vocab size is the same (32k) but the merge quality differs. With less training data, our 32,000 merge rules are learned from fewer examples. Some merges may be driven by noise or overrepresented articles rather than truly frequent patterns. GPT-2's merges are statistically much more robust.

3. GPT-2 was trained on broader text, not just Wikipedia. WebText includes news, blogs, forums, Reddit links — a much wider variety of English. This means GPT-2's vocabulary covers more general English patterns. Our tokenizer over-specializes on Wikipedia style (formal, encyclopedic) and may miss common informal patterns that still appear in the eval text.

4. GPT-2 uses 50,257 tokens, not 32,000. GPT-2's vocabulary is actually ~56% larger than ours. More vocabulary entries = more words get their own token = fewer splits = lower fertility. This is probably the single biggest factor


### ✏️ Exercise 2

1. Change `vocab_size` to `8_000` and retrain. How does fertility change? What tokens disappear?
2. Try encoding a French or Spanish sentence with your tokenizer. What happens? Why?
3. Look at `wiki_tokenizer.model.merges[:20]`. These are the first 20 merge rules learned. What patterns do you notice?

In [ ]:
# Your answers here

# Hint for Q3:
# The `merges` attribute is not directly accessible from wiki_tokenizer.model.
# To inspect the merges, you need to load them from the tokenizer's configuration file.
import json

# Load the tokenizer configuration from the saved file
with open('/tmp/wiki_tokenizer.json', 'r') as f:
    tokenizer_config = json.load(f)

# Access the merges list from the loaded configuration
merges_list = tokenizer_config['model']['merges']
print("First 20 merge rules learned:")
for merge_rule in merges_list[:20]:
    print(f"  {merge_rule}")

<details>
<summary>Click to reveal: Analysis of the first merge rules</summary>

The code displayed the first 20 merge rules learned by the BPE tokenizer we trained. Here's what we can observe:

Initial merges are often combinations of very frequent characters, particularly:

* **Characters preceded by a space (Ġ)**: We see `['Ġ', 't']`, `['Ġ', 'a']`, `['Ġ', ',']`, `['Ġ', '.']`, `['Ġ', 's']`, `['Ġ', 'o']`, `['Ġ', 'w']`, `['Ġ', 'c']`. This indicates that the tokenizer quickly learns to recognize word beginnings or short words as distinct units when they follow a space.
* **Common letter combinations**: Pairs like `['h', 'e']` (to form 'he' or 'the'), `['i', 'n']`, `['e', 'r']`, `['o', 'n']`, `['r', 'e']`, `['e', 'd']`, `['n', 'd']`, `['a', 't']`, `['o', 'r']`, `['e', 'n']`, `['i', 't']` appear very early. These are very frequent character bigrams in the English language.
* **Prefix merges with spaces**: We note `['Ġt', 'he']`, which shows that the tokenizer first merged Ġ and t to form Ġt, then this new Ġt unit was merged with he to form Ġthe, a very common word.

These patterns illustrate the greedy nature of the BPE algorithm, which combines the most frequent pairs of symbols to build longer tokens, starting with the most basic character combinations and progressing to more complex subwords.

</details>

---
# Part 3 — Pre-trained Tokenizers in Depth: Padding, Batching, Masks

In practice you'll almost always use a **pre-trained tokenizer** from HuggingFace.  
This section covers everything you need to feed text into a real model correctly.

We'll use **GPT-2** throughout, a decoder-only model.  
Key differences to note:
- GPT-2 has **no** `[CLS]` or `[SEP]` — it's causal LM, no classification tokens needed
- GPT-2 was **not trained with padding** (it processes variable-length sequences)
- For batching, we must add padding ourselves

### 3.1 — Special Tokens

Every model family defines **special tokens** — reserved token IDs with specific meanings that the model learned during pre-training:

| Token | Used by | Purpose |
|-------|---------|---------|
| `[CLS]` | BERT | Placed at the start of input. Its hidden state becomes the "sentence embedding" for classification. |
| `[SEP]` | BERT | Separates sentence pairs and marks end of input. |
| `[MASK]` | BERT | Placeholder for tokens the model must predict during masked LM training. |
| `[PAD]` | BERT, and manually added for GPT-2 | Fills shorter sequences in a batch. Ignored via the attention mask. |
| `<\|endoftext\|>` | GPT-2 | Marks the end of a document. GPT-2's only built-in special token. |
| `<s>`, `</s>` | LLaMA, Mistral | Beginning and end of sequence markers. |

GPT-2 was not designed for batching, so it has **no pad token by default** — we need to assign one manually (typically reusing `eos_token`).

Let's inspect what each tokenizer actually has:

In [ ]:
tokenizer = AutoTokenizer.from_pretrained("gpt2")

# GPT-2 only has one special token: end-of-text
print("All special tokens:")
print(tokenizer.special_tokens_map)
print()

# GPT-2 has no pad token by default — we need to add one for batching
# Common convention: reuse eos_token as pad_token
tokenizer.pad_token = tokenizer.eos_token

print(f"eos_token         : {repr(tokenizer.eos_token)}  (id={tokenizer.eos_token_id})")
print(f"pad_token (added) : {repr(tokenizer.pad_token)}  (id={tokenizer.pad_token_id})")
print()

# Compare with BERT which has more special tokens
bert = AutoTokenizer.from_pretrained("bert-base-uncased")
print("BERT special tokens:")
print(bert.special_tokens_map)

### 3.2 — Encoding a Single Sequence

When you call `tokenizer(text)`, it returns a **BatchEncoding** — a dictionary-like object containing:
- `input_ids`: the token IDs (integers indexing into the vocabulary)
- `attention_mask`: 1 for real tokens, 0 for padding (all 1s here since there's no padding yet)

You can decode IDs back to text with `tokenizer.decode()`, and inspect individual tokens with `convert_ids_to_tokens()`.

In [ ]:
text = "Tokenizers are the unsung heroes of NLP."

# The __call__ interface returns a BatchEncoding (dict-like)
encoding = tokenizer(text, return_tensors="pt")

print("Keys in encoding:", list(encoding.keys()))
print()
print("input_ids   :", encoding["input_ids"])
print("attention_mask:", encoding["attention_mask"])
print()

# Decode back
decoded = tokenizer.decode(encoding["input_ids"][0])
print("Decoded:", repr(decoded))
print()

# Individual tokens
tokens = tokenizer.convert_ids_to_tokens(encoding["input_ids"][0])
print("Tokens:", tokens)

### 3.3 — Truncation

Models have a **maximum context length** (`model_max_length`).  
Any sequence longer than this must be truncated.

In [ ]:
print(f"GPT-2 max context: {tokenizer.model_max_length} tokens")

# Generate a very long text
long_text = "This is a very important sentence. " * 200  # ~1400 tokens

# Without truncation
enc_full = tokenizer(long_text)
print(f"\nWithout truncation : {len(enc_full['input_ids'])} tokens")

# With truncation (clips to model_max_length)
enc_trunc = tokenizer(long_text, truncation=True)
print(f"With truncation    : {len(enc_trunc['input_ids'])} tokens")

# Explicit max_length
enc_custom = tokenizer(long_text, truncation=True, max_length=128)
print(f"Custom max_length=128: {len(enc_custom['input_ids'])} tokens")

### 3.4 — Building a Batch with Padding 🏗️

**The problem:** sentences in a batch have different lengths.
Tensors must be rectangular → we need to add **padding tokens** to shorter sequences.

```
Sentence 1: [The] [cat] [sat]                → 3 tokens
Sentence 2: [The] [quick] [brown] [fox]       → 4 tokens  
Sentence 3: [Hi]                              → 1 token

After padding (pad_right):
[[The] [cat]   [sat]   [PAD] ]
[[The] [quick] [brown] [fox] ]
[[Hi]  [PAD]   [PAD]   [PAD] ]
```

**The attention mask** tells the model which tokens are real (1) vs padding (0).

**Why does this matter inside the model?**
- During self-attention, the model computes attention scores between **all** positions. Without the mask, real tokens would attend to padding tokens, mixing meaningless values into hidden states.
- During training, the loss function must **ignore** padding positions. Otherwise the model would be penalized for failing to "predict" padding, corrupting the gradient signal.
- In short: the attention mask is not just bookkeeping. It directly prevents **corrupted representations** and **wrong loss values**.

In [ ]:
# ── Step 1: Encode each sentence independently ──────────────────────────────
sentences = [
    "The cat sat on the mat.",
    "Tokenization is surprisingly interesting and has many edge cases.",
    "Hi!",
]

print("=== STEP 1: Individual encodings (no padding) ===")
individual = []
for sent in sentences:
    enc = tokenizer(sent)
    individual.append(enc["input_ids"])
    tokens = tokenizer.convert_ids_to_tokens(enc["input_ids"])
    print(f"  {len(enc['input_ids']):2d} tokens | {tokens}")

We'll build a batch in 5 steps: first manually, then using the tokenizer's built-in batching. This way you understand exactly what happens under the hood.

**Step 1:** Encode each sentence independently. Notice they all have different lengths.

In [ ]:
# ── Step 2: Manually pad to max length ──────────────────────────────────────
print("\n=== STEP 2: Manual padding ===")
max_len = max(len(ids) for ids in individual)
pad_id  = tokenizer.pad_token_id

padded_ids   = []
padded_masks = []

for ids in individual:
    n_pad  = max_len - len(ids)
    padded = ids + [pad_id] * n_pad        # right-pad
    mask   = [1] * len(ids) + [0] * n_pad  # 1=real, 0=padding
    padded_ids.append(padded)
    padded_masks.append(mask)

print(f"Max length in batch: {max_len}")
print(f"\nPadded input_ids:")
for row in padded_ids:
    print(" ", row)

print(f"\nAttention masks:")
for row in padded_masks:
    print(" ", row)

**Step 2:** Manually pad all sequences to the length of the longest one. Each shorter sequence gets pad tokens appended, and its attention mask gets 0s in those positions.

In [ ]:
# ── Step 3: Let the tokenizer handle it automatically ───────────────────────
print("\n=== STEP 3: Automatic batching with the tokenizer ===")

batch = tokenizer(
    sentences,
    padding=True,         # pad to the longest sequence in the batch
    truncation=True,      # truncate if longer than max_length
    max_length=32,
    return_tensors="pt",  # return PyTorch tensors
)

print("input_ids shape:", batch["input_ids"].shape)    # (batch_size, seq_len)
print("attention_mask shape:", batch["attention_mask"].shape)
print()
print("input_ids:\n", batch["input_ids"])
print()
print("attention_mask:\n", batch["attention_mask"])

**Step 3:** The tokenizer can do all of this automatically when you pass a list of sentences with `padding=True`. It returns PyTorch tensors ready for the model.

In [ ]:
# ── Step 4: Padding strategies ───────────────────────────────────────────────
print("=== Padding strategies ===")

# padding='max_length' pads ALL sequences to max_length (not just the longest)
batch_max = tokenizer(sentences, padding="max_length", max_length=20, return_tensors="pt")
print("padding='max_length', max_length=20")
print("Shape:", batch_max["input_ids"].shape)
print(batch_max["input_ids"])

print()

# padding=True pads to longest in batch (most efficient for training)
batch_longest = tokenizer(sentences, padding=True, return_tensors="pt")
print("padding=True (longest in batch)")
print("Shape:", batch_longest["input_ids"].shape)
print(batch_longest["input_ids"])

**Step 4: Padding strategies.** Two main options:
- `padding=True` (or `"longest"`) —> pad to the longest sequence **in the current batch**. Most efficient.
- `padding="max_length"` —> pad every sequence to `max_length`. Produces uniform shapes but wastes compute on short batches.

In [ ]:
# ── Step 5: Padding side (left vs right) ────────────────────────────────────
# For DECODER-ONLY models (GPT-2, LLaMA): use LEFT padding during generation
#   → so all sequences end at the same position (last real token is at the right edge)
# For ENCODER models (BERT): right padding is fine

print("=== Padding side matters for generation ===")
print()

tokenizer.padding_side = "right"
batch_right = tokenizer(sentences[:2], padding=True, return_tensors="pt")
print("Right padding (default for most models):")
print(batch_right["input_ids"])
print(batch_right["attention_mask"])

print()

tokenizer.padding_side = "left"
batch_left = tokenizer(sentences[:2], padding=True, return_tensors="pt")
print("Left padding (preferred for decoder generation):")
print(batch_left["input_ids"])
print(batch_left["attention_mask"])

# Reset to right for rest of notebook
tokenizer.padding_side = "right"

**Step 5: Padding side — left vs right.**

This matters for **decoder-only models** (GPT-2, LLaMA) during **generation**:
- With **right** padding, the last real token is at a different position in each sequence —> the model doesn't know where to start generating.
- With **left** padding, all sequences end at the right edge —> the model can generate from the same position for every sequence in the batch.

### 3.5 — Offset Mappings (character → token alignment)

Offset mappings let you map each token back to its **exact character positions** in the original string. This is essential for tasks where you need to connect model predictions to specific spans of text.

Two important use cases:
- **Named Entity Recognition (NER)** — the task of identifying and classifying named entities (people, organizations, locations, etc.) in text. For example, in *"Barack Obama visited MIT"*, a NER model should detect *"Barack Obama"* as a **Person** and *"MIT"* as an **Organization**.
- **Question Answering (QA)** — extracting an answer span from a passage. The model predicts start/end token positions, and offsets convert those back to character positions in the original text.

In [ ]:
text = "Paris is the capital of France."

enc = tokenizer(text, return_offsets_mapping=True)

tokens  = tokenizer.convert_ids_to_tokens(enc["input_ids"])
offsets = enc["offset_mapping"]

print(f"{'Token':<15} {'Char span':<12} {'Original chars'}")
print("─" * 45)
for token, (start, end) in zip(tokens, offsets):
    original = text[start:end] if end > start else ""
    print(f"{repr(token):<15} [{start:2d}, {end:2d}]      {repr(original)}")

### 3.6 — Tokenizing a Full Dataset

In practice, you never tokenize one sentence at a time. The standard pattern uses HuggingFace's `.map()` method, which:
1. Applies your tokenization function to every example (or batch of examples) in the dataset
2. Adds the new columns (`input_ids`, `attention_mask`) alongside the original ones
3. Caches the result so you don't re-tokenize every time

This is much more efficient than looping through examples manually, and integrates directly with PyTorch's `DataLoader`.


In [ ]:
# Load a tiny split for demonstration
small_dataset = load_dataset("imdb", split="train[:200]")

def tokenize_function(examples):
    return tokenizer(
        examples["text"],
        truncation=True,
        max_length=128,
        padding="max_length",
    )

# .map() applies the function batched across the whole dataset
tokenized_dataset = small_dataset.map(tokenize_function, batched=True)

print("Dataset columns:", tokenized_dataset.column_names)
print("First example input_ids (first 20):", tokenized_dataset[0]["input_ids"][:20])
print("First example mask    (first 20):", tokenized_dataset[0]["attention_mask"][:20])

# Distribution of sequence lengths (before truncation would matter)
lengths = [sum(m) for m in tokenized_dataset["attention_mask"]]  # real tokens only
print(f"\nSequence lengths — min:{min(lengths)}, max:{max(lengths)}, "
      f"mean:{sum(lengths)/len(lengths):.1f}")

### ✏️ Exercise 3

1. Create a batch of 4 sentences of very different lengths. Inspect the `attention_mask`.
   Why is it critical that the model ignores pad tokens?
2. What happens if you try to decode a padded sequence with `tokenizer.decode(...)` without
   specifying `skip_special_tokens=True`?
3. For a GPT-2 model doing text generation, should you pad left or right? Why?
   (Hint: think about where the model predicts the *next* token from.)

In [ ]:
# Your answers here


---
# Part 4 — Integration into a Small LLM

### The key architectural insight

The tokenizer and model are **tightly coupled** through `vocab_size`:

```
text  →  [tokenizer]  →  token IDs  →  [embedding table]  →  vectors  →  [transformer]  →  [output projection]  →  logits over vocab
                                        shape: (V, d_model)                                  shape: (d_model, V)
```

Where `V = vocab_size`. **Changing the tokenizer means changing V, which means rebuilding both the embedding table and the output layer — i.e. the model must be retrained from scratch.**

### 4.1 — Load GPT-2 and Inspect the Architecture

Let's load the pre-trained GPT-2 model and examine the two components that directly depend on `vocab_size`:
- **`wte`** (Word Token Embedding): a matrix of shape `(vocab_size, d_model)` that maps each token ID to a dense vector.
- **`lm_head`**: a linear layer of shape `(d_model, vocab_size)` that projects hidden states back to logits over the vocabulary.

GPT-2 **ties** these two matrices (they share the same weights), which halves the number of parameters dedicated to vocabulary.

In [ ]:
model = GPT2LMHeadModel.from_pretrained("gpt2")
model.eval()

tokenizer = AutoTokenizer.from_pretrained("gpt2")
tokenizer.pad_token = tokenizer.eos_token

print(model.config)
print()
print(f"Total parameters: {sum(p.numel() for p in model.parameters()):,}")

Now let's look at the layers whose dimensions are directly determined by `vocab_size`.

These are the **only two places** in the model architecture where the vocabulary size appears. Every other layer (attention, feed-forward, layer norms) is independent of the tokenizer. This is why changing the tokenizer requires changing these specific layers and losing the learned weights.


In [ ]:
# The token embedding table: shape = (vocab_size, hidden_size)
wte = model.transformer.wte  # Word Token Embedding
wpe = model.transformer.wpe  # Word Position Embedding

print("Token embedding (wte) shape:", wte.weight.shape)
print("  → rows = vocab_size =", wte.weight.shape[0])
print("  → cols = d_model    =", wte.weight.shape[1])
print()
print("Position embedding (wpe) shape:", wpe.weight.shape)
print("  → rows = max_positions =", wpe.weight.shape[0])
print()

# The output language model head: maps hidden states → logits over vocab
lm_head = model.lm_head
print("LM head (lm_head) weight shape:", lm_head.weight.shape)
print("  → Note: same shape as wte — GPT-2 ties input/output embeddings!")
print()
print(f"Tokenizer vocab size: {tokenizer.vocab_size}")
print(f"Model vocab size:     {model.config.vocab_size}")
print(f"Match: {tokenizer.vocab_size == model.config.vocab_size} ✅")

Notice that `wte` and `lm_head` have the **same shape**. GPT-2 uses weight tying, meaning they share the exact same tensor in memory. The tokenizer's vocab size and the model's vocab size must match exactly.

### 4.2 — The Full Pipeline: Encode → Forward Pass → Decode

Let's trace a single prompt through **every step** of the model pipeline:
1. **Tokenize** — convert text to token IDs
2. **Embed** — look up each ID in the embedding table, add positional embeddings
3. **Forward pass** — run through all transformer layers to get logits
4. **Predict** — pick the most likely next token from the logits at the last position
5. **Decode** — convert the predicted token ID back to text

In [ ]:
prompt = "The future of artificial intelligence is"

# ── Step 1: Tokenize ─────────────────────────────────────────────────────────
inputs = tokenizer(prompt, return_tensors="pt")
input_ids = inputs["input_ids"]           # shape: (1, seq_len)
print("Step 1 — Token IDs:", input_ids)
print("         Tokens    :", tokenizer.convert_ids_to_tokens(input_ids[0]))

# ── Step 2: Embedding lookup ─────────────────────────────────────────────────
with torch.no_grad():
    token_embeddings = model.transformer.wte(input_ids)    # (1, seq_len, 768)
    pos_ids          = torch.arange(input_ids.shape[1]).unsqueeze(0)
    pos_embeddings   = model.transformer.wpe(pos_ids)      # (1, seq_len, 768)
    hidden_states    = token_embeddings + pos_embeddings   # (1, seq_len, 768)

print(f"\nStep 2 — Embeddings shape: {hidden_states.shape}")
print(f"          (batch=1, seq_len={input_ids.shape[1]}, d_model=768)")

# ── Step 3: Full forward pass ─────────────────────────────────────────────────
with torch.no_grad():
    outputs = model(**inputs)

logits = outputs.logits   # shape: (1, seq_len, vocab_size)
print(f"\nStep 3 — Logits shape: {logits.shape}")
print(f"          (batch=1, seq_len={input_ids.shape[1]}, vocab_size=50257)")

# ── Step 4: Predict next token ────────────────────────────────────────────────
# Logits at the LAST position = distribution over what comes next
next_token_logits = logits[0, -1, :]          # shape: (vocab_size,)
next_token_probs  = torch.softmax(next_token_logits, dim=-1)

top5 = torch.topk(next_token_probs, 5)
print("\nStep 4 — Top 5 predicted next tokens:")
for prob, idx in zip(top5.values, top5.indices):
    token = tokenizer.decode([idx.item()])
    print(f"  {prob:.4f}  |  id={idx.item():5d}  |  {repr(token)}")

# ── Step 5: Decode ────────────────────────────────────────────────────────────
best_next_id    = top5.indices[0].unsqueeze(0).unsqueeze(0)
full_ids        = torch.cat([input_ids, best_next_id], dim=1)
decoded         = tokenizer.decode(full_ids[0], skip_special_tokens=True)
print(f"\nStep 5 — Greedy next token appended:")
print(f"  '{decoded}'")

The output logits have shape `(batch, seq_len, vocab_size)` — at each position, the model produces a probability distribution over all 50,257 tokens. The **last position** tells us what the model predicts should come next.

The greedy approach (picking the most probable token) is simple but can produce repetitive text. Next, let's build a full autoregressive loop.

### 4.3 — Greedy Decoding Loop (manual autoregressive generation)

**Autoregressive generation** means producing text one token at a time. At each step:
1. Feed all tokens so far into the model
2. Take the logits at the **last position** (the model's prediction for what comes next)
3. Pick the token with the highest probability (greedy) or sample from the distribution
4. Append that token and repeat

This is intentionally simple — greedy decoding always picks the most probable token, which tends to produce repetitive text. Real generation uses sampling strategies (temperature, top-k, top-p) to add diversity.


In [ ]:
def greedy_generate(prompt, model, tokenizer, max_new_tokens=30):
    """Manual greedy decoding — one token at a time."""
    input_ids = tokenizer(prompt, return_tensors="pt")["input_ids"]
    generated = input_ids.clone()

    print(f"Prompt: '{prompt}'")
    print(f"Generating {max_new_tokens} tokens...\n")

    for step in range(max_new_tokens):
        with torch.no_grad():
            logits = model(generated).logits

        # Pick the token with the highest logit at the last position
        next_id = logits[0, -1, :].argmax().unsqueeze(0).unsqueeze(0)
        generated = torch.cat([generated, next_id], dim=1)

        # Decode just the new token for display
        new_token = tokenizer.decode(next_id[0])
        print(f"  step {step+1:2d} → {repr(new_token)}")

        # Stop at EOS
        if next_id.item() == tokenizer.eos_token_id:
            print("  [EOS reached]")
            break

    final = tokenizer.decode(generated[0], skip_special_tokens=True)
    print(f"\nFull output: '{final}'")
    return final

greedy_generate("Tokenization is the process of", model, tokenizer, max_new_tokens=20)

### 4.4 — Using `.generate()` (the real API)

In practice, you never write a manual generation loop. HuggingFace's `.generate()` method handles everything, including **sampling** (randomness), **temperature** (controlling randomness), **top-k/top-p filtering**, and **beam search**.

The key parameters:
- `do_sample=True` — enable random sampling instead of greedy decoding
- `temperature` — lower = more deterministic, higher = more creative/random
- `top_k` — only sample from the top k most likely tokens at each step

In [ ]:
# The .generate() method handles all of this + sampling + beam search etc.
inputs = tokenizer("The history of Wikipedia started when", return_tensors="pt")

with torch.no_grad():
    output_ids = model.generate(
        **inputs,
        max_new_tokens=50,
        do_sample=True,       # sampling (not greedy)
        temperature=0.8,      # controls randomness
        top_k=50,             # only sample from top 50 tokens
        pad_token_id=tokenizer.eos_token_id,
    )

result = tokenizer.decode(output_ids[0], skip_special_tokens=True)
print(result)

### 4.5 — What Happens If You Change the Tokenizer?

This is one of the most common mistakes when working with LLMs. The tokenizer and model are trained **together** and the embedding table has one row per token in the vocabulary. If you swap the tokenizer:

- The same text produces **different token IDs** → the model looks up the **wrong embeddings**
- The vocabulary size may differ → the embedding matrix has the **wrong shape**
- Even if the vocab size matches by coincidence, the ID-to-token mapping is different

Let's demonstrate this concretely by comparing GPT-2's vocabulary with our custom tokenizer's vocabulary.


In [ ]:
from transformers import GPT2Config

# Our Wikipedia tokenizer has 32,000 tokens
# GPT-2 has 50,257 tokens
# What if we tried to use GPT-2's architecture with our tokenizer?

our_vocab_size = wiki_tokenizer.get_vocab_size()  # 32,000
gpt2_vocab_size = model.config.vocab_size          # 50,257

print(f"GPT-2 vocab size:      {gpt2_vocab_size:,}")
print(f"Our wiki vocab size:   {our_vocab_size:,}")
print()

# Build a GPT-2-style config with our vocab size
our_config = GPT2Config(
    vocab_size=our_vocab_size,
    n_embd=768,
    n_layer=12,
    n_head=12,
)

# Randomly initialized model (weights are random — it would need training!)
our_model = GPT2LMHeadModel(our_config)

print("GPT-2 original embedding table:")
print("  wte shape:", model.transformer.wte.weight.shape)
print()
print("Our model embedding table (with our vocab):")
print("  wte shape:", our_model.transformer.wte.weight.shape)
print()
print("⚠️  The embedding tables have different sizes.")
print("   GPT-2's trained weights CANNOT be loaded into our model.")
print("   The model would need to be trained from scratch with our tokenizer.")

The embedding tables have **different sizes**: GPT-2 has 50,257 rows, our model has 32,000 rows. You **cannot** load GPT-2's trained weights into a model built for a different vocab size. The model would need to be retrained from scratch with the new tokenizer.

### ✏️ Exercise 4

1. What are the two layers in GPT-2 whose sizes are directly determined by `vocab_size`?
   Find them in `model.named_parameters()` and print their shapes.

2. The GPT-2 model uses **weight tying**: the `lm_head` shares weights with `wte`.  
   Verify this with `model.transformer.wte.weight is model.lm_head.weight`.  
   Why is weight tying useful?

3. Run generation with `temperature=0.1` and `temperature=2.0`. What do you observe?
   Explain why in terms of the softmax over logits.

In [ ]:
# Your answers here

# Q1 hint:
for name, param in model.named_parameters():
    if param.shape[0] == model.config.vocab_size or (len(param.shape) > 1 and param.shape[1] == model.config.vocab_size):
        print(f"{name}: {param.shape}")

---
# Part 5 — From Tokenization to Training-Ready Data

You now know how tokenizers work and how to use them with models.
This part bridges the gap between "I can tokenize a sentence" and "I can prepare data that a model actually trains on."

We'll cover:
- Text normalization choices
- Special token conventions across models
- Task-specific label construction (classification, causal LM, NER)
- Dynamic padding with data collators
- Preprocessing discipline and common bugs

### 5.1 — Text Normalization & Cleaning

Before tokenization, raw text often goes through **normalization**:
- **Lowercasing**: BERT's `uncased` model lowercases everything; `cased` models preserve case
- **Unicode normalization** (NFC/NFKC): e.g. `é` (single char) vs `e` + `´` (two chars) — these look identical but are different bytes
- **Whitespace normalization**: collapse multiple spaces, strip leading/trailing
- **Punctuation handling**: some tokenizers split on punctuation, others don't
- **Domain-specific**: remove URLs, emails, HTML tags, or emojis — **only if irrelevant to the task**

> **Key takeaway:** Cleaning is **task-dependent**. A sentiment model might strip URLs; a web-safety classifier should keep them. Always document your cleaning choices.

In [ ]:
# --- Unicode normalization matters ---
text_nfc  = "caf\u00e9"              # é as a single character (NFC)
text_nfd  = "cafe\u0301"             # e + combining accent (NFD)

print(f"NFC: {repr(text_nfc)}  →  len = {len(text_nfc)}")
print(f"NFD: {repr(text_nfd)}  →  len = {len(text_nfd)}")
print(f"Look the same? '{text_nfc}' vs '{text_nfd}'")
print(f"Are equal?      {text_nfc == text_nfd}")
print()

# After normalization they become identical
normalized = unicodedata.normalize("NFC", text_nfd)
print(f"After NFC normalization: {repr(normalized)}, equal = {text_nfc == normalized}")
print()

# --- Lowercasing changes tokenization ---
mixed_case = "The CRISPR-Cas9 technique revolutionized Gene Editing."

bert_uncased = AutoTokenizer.from_pretrained("bert-base-uncased")
bert_cased   = AutoTokenizer.from_pretrained("bert-base-cased")

tokens_uncased = bert_uncased.tokenize(mixed_case)
tokens_cased   = bert_cased.tokenize(mixed_case)

print(f"BERT uncased ({len(tokens_uncased)} tokens): {tokens_uncased}")
print(f"BERT cased   ({len(tokens_cased)} tokens): {tokens_cased}")
print()
print("Notice: uncased model lowercases first, so 'CRISPR' and 'crispr' are the same.")
print("Cased model preserves case — important for NER (proper nouns) and code.")

The cell above showed how two visually identical strings (`"café"`) can have different byte representations depending on Unicode normalization. BERT's tokenizer handles this internally, but it's good practice to normalize your data before tokenization to avoid inconsistencies.

Now let's see the effect of **whitespace normalization**, another common source of wasted tokens:


In [ ]:
# --- Whitespace normalization effect ---
messy_text = "  Hello   world!   Multiple    spaces   everywhere.  "
clean_text = " ".join(messy_text.split())

print(f"Messy: {repr(messy_text)}")
print(f"Clean: {repr(clean_text)}")
print()

gpt2_messy = gpt2_tok.tokenize(messy_text)
gpt2_clean = gpt2_tok.tokenize(clean_text)
print(f"GPT-2 on messy ({len(gpt2_messy)} tokens): {gpt2_messy}")
print(f"GPT-2 on clean ({len(gpt2_clean)} tokens): {gpt2_clean}")
print()
print("Extra whitespace wastes tokens and may confuse models trained on clean text.")

**Results:** Extra whitespace produces extra tokens —> wasted compute and potentially confusing for models that were trained on clean text. A simple `" ".join(text.split())` can significantly reduce token count.

The next section shows how **special tokens** differ across model families, a common source of silent bugs.

### 5.2 — Special Token Behavior Across Models

Different models use different special-token conventions. Getting this wrong is a **silent bug**: the model won't crash, but it will produce degraded results.

In [ ]:
# --- Compare special token conventions ---
gpt2  = AutoTokenizer.from_pretrained("gpt2")
bert  = AutoTokenizer.from_pretrained("bert-base-uncased")

print("=" * 60)
print("Special token comparison: GPT-2 vs BERT")
print("=" * 60)

for name, tok in [("GPT-2", gpt2), ("BERT", bert)]:
    print(f"\n{name}:")
    for attr in ["bos_token", "eos_token", "cls_token", "sep_token", "pad_token", "mask_token"]:
        val = getattr(tok, attr, None)
        print(f"  {attr:15s} = {repr(val)}")

# --- Encode a single sentence ---
text = "Paris is the capital of France."

print("\n" + "=" * 60)
print("Single sentence encoding")
print("=" * 60)

gpt2_ids = gpt2(text)["input_ids"]
bert_ids = bert(text)["input_ids"]

print(f"\nGPT-2: {gpt2.convert_ids_to_tokens(gpt2_ids)}")
print(f"  → No special tokens added (GPT-2 adds nothing by default)")

print(f"\nBERT:  {bert.convert_ids_to_tokens(bert_ids)}")
print(f"  → Wrapped with [CLS] ... [SEP]")

# --- Encode a sentence pair ---
text_a = "Paris is in France."
text_b = "Berlin is in Germany."

print("\n" + "=" * 60)
print("Sentence pair encoding")
print("=" * 60)

bert_pair = bert(text_a, text_b)["input_ids"]
print(f"\nBERT:  {bert.convert_ids_to_tokens(bert_pair)}")
print(f"  → [CLS] sent_A [SEP] sent_B [SEP]")

gpt2_pair = gpt2(text_a, text_b)["input_ids"]
print(f"\nGPT-2: {gpt2.convert_ids_to_tokens(gpt2_pair)}")
print(f"  → Simply concatenated — no separator (causal models don't use pairs)")

**Key observations from the output above:**
- **GPT-2** has only `eos_token` — no `cls_token`, no `sep_token`, no `pad_token`, no `mask_token`. It was designed for open-ended text generation, not classification or masked language modeling.
- **BERT** has a full set of special tokens because it was designed for classification (`[CLS]`), sentence-pair tasks (`[SEP]`), and masked prediction (`[MASK]`).
- For **single sentences**, BERT wraps with `[CLS]...[SEP]` while GPT-2 adds nothing.
- For **sentence pairs**, BERT inserts `[SEP]` between them. GPT-2 just concatenates because it was never designed for pair tasks.

### Mini-Exercise

Encode the sentence pair `("The movie was great", "I loved every minute")` with both BERT and GPT-2.
How does each tokenizer format the pair? What tokens are inserted between the two sentences?

Try also with Mistral. Does it behave like BERT or GPT-2?

### 5.3 — From Raw Text to Training Batch

The tokenizer's output is not yet what the model consumes during training.
Each task type requires a different **label strategy** and **data format**.

We'll walk through three common scenarios:
- **A.** Text Classification (sentiment analysis)
- **B.** Causal Language Modeling (GPT-style pretraining)
- **C.** Token Classification (Named Entity Recognition)

#### A. Text Classification (e.g., sentiment analysis)

**Goal:** tokenize text, preserve the label, build a batch.
- `input_ids`: token IDs from the tokenizer
- `attention_mask`: 1 for real tokens, 0 for padding
- `labels`: a **single integer** per example (the class index)

In [ ]:
# --- Text Classification: from raw examples to a training batch ---
clf_tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

# Raw examples: text + label
examples = [
    {"text": "This movie was absolutely fantastic!",          "label": 1},  # positive
    {"text": "Terrible acting and boring plot.",               "label": 0},  # negative
    {"text": "A masterpiece of modern cinema, truly moving.", "label": 1},  # positive
]

# Step 1: Tokenize all texts
texts  = [ex["text"] for ex in examples]
labels = [ex["label"] for ex in examples]

batch = clf_tokenizer(
    texts,
    padding=True,
    truncation=True,
    max_length=32,
    return_tensors="pt",
)

# Step 2: Add labels — they are NOT produced by the tokenizer
batch["labels"] = torch.tensor(labels)

print("Batch contents:")
for key, val in batch.items():
    print(f"  {key:15s} shape={str(val.shape):12s} dtype={val.dtype}")
print()
print("input_ids:\n", batch["input_ids"])
print("attention_mask:\n", batch["attention_mask"])
print("labels:", batch["labels"])
print()
print("Key insight: the tokenizer handles text → token IDs.")
print("Labels are managed separately and added to the batch manually.")

**What happened above:** We tokenized 3 movie reviews, padded them to equal length, and added labels as a separate tensor. The tokenizer only handles the text and **labels are your responsibility**.

Now let's feed this batch into a real classification model to see the full loop:

In [ ]:
# --- Feed the batch to a classification model ---
clf_model = AutoModelForSequenceClassification.from_pretrained(
    "bert-base-uncased", num_labels=2
)
clf_model.eval()

with torch.no_grad():
    outputs = clf_model(**batch)

print(f"Loss:   {outputs.loss.item():.4f}")
print(f"Logits: {outputs.logits}")
print(f"Logits shape: {outputs.logits.shape}  →  (batch_size, num_labels)")
print()
print("The model receives input_ids + attention_mask + labels,")
print("and returns both logits (predictions) and loss (for training).")

**What the model returned:**
- `logits` — raw prediction scores for each class (shape: batch_size x num_labels). These would be passed through softmax to get probabilities.
- `loss` — cross-entropy loss between predictions and labels. This is what gets backpropagated during training.

The model is untrained (random weights), so the loss and predictions are meaningless, but the **data pipeline** is correct and ready for training.

#### B. Causal Language Modeling (e.g., GPT-2 pretraining)

**Key insights:**
- `labels = input_ids` — the model predicts the next token at each position
- The model **internally shifts** labels left by one position — you do **not** shift them yourself
- Padding tokens must be masked with **`-100`** in the labels so the loss ignores them

> `-100` is PyTorch's convention for "ignore this position in cross-entropy loss".

In [ ]:
# --- Causal LM: building labels with -100 masking ---
lm_tokenizer = AutoTokenizer.from_pretrained("gpt2")
lm_tokenizer.pad_token = lm_tokenizer.eos_token

sentences = [
    "The cat sat on the mat.",
    "Hi!",
]

batch = lm_tokenizer(
    sentences,
    padding=True,
    truncation=True,
    max_length=32,
    return_tensors="pt",
)

# Labels = input_ids, but with -100 where attention_mask == 0 (padding)
labels = batch["input_ids"].clone()
labels[batch["attention_mask"] == 0] = -100

print("input_ids:\n", batch["input_ids"])
print()
print("attention_mask:\n", batch["attention_mask"])
print()
print("labels (padding masked with -100):\n", labels)
print()
print("Notice: -100 appears exactly where attention_mask is 0.")
print("The model will NOT compute loss on these positions.")

**Understanding the labels tensor:**
- Where `attention_mask == 1` (real tokens): `labels = input_ids` — the model should predict these tokens.
- Where `attention_mask == 0` (padding): `labels = -100` — PyTorch's cross-entropy loss **ignores** these positions entirely.

Without this masking, the model would waste capacity trying to predict padding tokens, corrupting the training signal. Let's verify this matters by comparing loss with and without masking:

In [ ]:
# --- Feed to GPT-2 and verify loss ---
with torch.no_grad():
    # With correct -100 masking
    outputs_masked = model(
        input_ids=batch["input_ids"],
        attention_mask=batch["attention_mask"],
        labels=labels,
    )
    # Without masking (BUG: loss is computed on pad tokens too)
    outputs_unmasked = model(
        input_ids=batch["input_ids"],
        attention_mask=batch["attention_mask"],
        labels=batch["input_ids"],  # no -100 masking!
    )

print(f"Loss with -100 masking:    {outputs_masked.loss.item():.4f}  (correct)")
print(f"Loss without masking:      {outputs_unmasked.loss.item():.4f}  (WRONG — includes pad tokens)")
print()
print("The unmasked loss is different because the model is being penalized")
print("for failing to predict padding tokens, which corrupts the training signal.")

**Result:** The two losses are different. The unmasked loss includes gradients from padding positions, which would push the model to memorize pad tokens instead of learning language patterns. Always mask padding in labels with `-100`.

#### C. Token Classification (Named Entity Recognition)

**What is NER?** Named Entity Recognition is the task of labeling each word in a sentence with an entity type. For example:

```
Barack   Obama   visited   the   Massachusetts   Institute   of   Technology   .
B-PER    I-PER   O         O     B-ORG           I-ORG        I-ORG I-ORG      O
```

Labels follow the **BIO scheme**: `B-` = beginning of an entity, `I-` = inside (continuation), `O` = outside (not an entity).

**The alignment problem:** NER labels are defined **per word**, but tokenizers split words into **subtokens**. One word may become 2, 3, or more tokens. The convention is:
- Assign the label to the **first** subtoken of each word
- Set all continuation subtokens to **`-100`** (ignored in loss)

In [ ]:
# --- NER: aligning word-level labels to subtoken-level labels ---
ner_tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

# NER example: words and their BIO labels
words  = ["Barack",  "Obama",  "visited", "the", "Massachusetts", "Institute", "of", "Technology", "."]
labels = ["B-PER",   "I-PER",  "O",       "O",   "B-ORG",         "I-ORG",     "I-ORG", "I-ORG",  "O"]

# Label-to-ID mapping
label2id = {"O": 0, "B-PER": 1, "I-PER": 2, "B-ORG": 3, "I-ORG": 4}
label_ids = [label2id[l] for l in labels]

# Tokenize with is_split_into_words=True (input is already split into words)
encoding = ner_tokenizer(
    words,
    is_split_into_words=True,
    return_tensors="pt",
)

# word_ids() maps each token back to its original word index
word_ids = encoding.word_ids()

# Align labels: first subtoken gets the label, rest get -100
aligned_labels = []
previous_word_id = None
for word_id in word_ids:
    if word_id is None:
        # Special tokens ([CLS], [SEP])
        aligned_labels.append(-100)
    elif word_id != previous_word_id:
        # First subtoken of a new word → assign the word's label
        aligned_labels.append(label_ids[word_id])
    else:
        # Continuation subtoken → mask with -100
        aligned_labels.append(-100)
    previous_word_id = word_id

aligned_labels = torch.tensor([aligned_labels])

**How `word_ids()` works:** After tokenizing with `is_split_into_words=True`, the encoding tracks which original word each token came from. We use this mapping to:
- Assign the real label to the **first** subtoken of each word
- Set `-100` for all **continuation** subtokens (so the model isn't penalized for them)
- Set `-100` for **special tokens** (`[CLS]`, `[SEP]`) which aren't real words

Let's display the result as a table:

In [ ]:
# --- Display the alignment table ---
tokens = ner_tokenizer.convert_ids_to_tokens(encoding["input_ids"][0])
id2label = {v: k for k, v in label2id.items()}

print(f"{'Token':<18} {'Word ID':<10} {'Label ID':<10} {'Label'}")
print("─" * 55)
for tok, wid, lid in zip(tokens, word_ids, aligned_labels[0].tolist()):
    label_str = id2label.get(lid, "IGNORE") if lid != -100 else "—  (masked)"
    wid_str   = str(wid) if wid is not None else "—"
    print(f"{tok:<18} {wid_str:<10} {lid:<10} {label_str}")

print()
print("Notice:")
print("  - [CLS] and [SEP] → -100 (special tokens, not real words)")
print("  - 'massachusetts' splits into 'massachusetts' (1 token here, but try longer words!)")
print("  - First subtoken of each word gets the real label")
print("  - Continuation subtokens would get -100 (try 'antidisestablishmentarianism'!)")

### 5.4 — Dynamic Padding with Data Collators

So far we've padded to a fixed length or to the longest sequence in a batch manually.
In practice, HuggingFace provides **data collators** that handle padding **at batch creation time** — each batch is padded only to its longest member.

This is more efficient: a batch of short sentences wastes no compute on long padding tails.

In [ ]:
# --- DataCollatorWithPadding: dynamic padding for classification ---
dc_tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
collator = DataCollatorWithPadding(tokenizer=dc_tokenizer)

# Tokenize WITHOUT padding — the collator will pad at batch time
texts = [
    "Short.",
    "A medium-length sentence about tokenization.",
    "This is a much longer sentence that demonstrates how dynamic padding adapts to the longest element in each specific batch.",
    "Hi!",
    "Another relatively short example here.",
    "Deep learning models need properly formatted input data to train effectively.",
]

tokenized = [dc_tokenizer(t, truncation=True, max_length=64) for t in texts]

# Create a DataLoader with the collator
loader = DataLoader(tokenized, batch_size=3, collate_fn=collator)

print("Dynamic padding in action — each batch has a different length:\n")
for i, batch in enumerate(loader):
    print(f"  Batch {i+1}: input_ids shape = {batch['input_ids'].shape}")
    real_lengths = batch["attention_mask"].sum(dim=1).tolist()
    print(f"           Real sequence lengths: {real_lengths}")
    print(f"           Padded to: {batch['input_ids'].shape[1]}")
    print()

**Result:** Each batch is padded to a **different length** — batch 1 might be padded to 25 tokens, batch 2 to 15 tokens. This is more efficient than padding everything to a fixed max length, especially when sequence lengths vary a lot.

Now let's see the `DataCollatorForLanguageModeling`, which not only pads but also **automatically creates the labels tensor** with `-100` on padding positions:

In [ ]:
# --- DataCollatorForLanguageModeling: automatic label creation for causal LM ---
lm_collator_tok = AutoTokenizer.from_pretrained("gpt2")
lm_collator_tok.pad_token = lm_collator_tok.eos_token

lm_collator = DataCollatorForLanguageModeling(
    tokenizer=lm_collator_tok,
    mlm=False,  # causal LM, not masked LM
)

# Tokenize without padding
lm_texts = [
    "The cat sat on the mat.",
    "Hello!",
    "Artificial intelligence is transforming many industries today.",
]
lm_tokenized = [lm_collator_tok(t, truncation=True, max_length=32) for t in lm_texts]

# The collator handles both padding AND label creation
lm_batch = lm_collator(lm_tokenized)

print("input_ids:\n", lm_batch["input_ids"])
print()
print("labels (auto-created by collator):\n", lm_batch["labels"])
print()
print("Notice: the collator automatically set -100 on pad positions in labels!")
print("This saves you from manually creating the labels tensor.")

**Key benefit:** The LM collator handles both padding **and** label masking automatically. Compare the `labels` row with the `input_ids` row — they're identical on real token positions and `-100` on padding. This is exactly what we built manually in section 5.3B, but now it's done for us.

### 5.5 — Preprocessing Discipline: Train / Validation / Test

**Critical rule:** Your tokenizer should be trained **only on training data**.
If you train it on the full dataset (including validation/test), you introduce information leakage.

| Do | Don't |
|---|---|
| Train tokenizer on `train` split only | Train on the full dataset |
| Evaluate fertility on held-out text | Only check fertility on training text |
| Document all preprocessing choices | Apply ad-hoc cleaning without recording it |

A tokenizer trained on one domain may perform poorly on another — let's see this in action.

In [ ]:
# --- Domain mismatch: our wiki tokenizer on different domains ---

domain_samples = {
    "Wikipedia (in-domain)": (
        "The French Revolution was a period of radical political and societal change "
        "in France that began with the Estates General of 1789 and ended with the "
        "formation of the French Consulate in November 1799."
    ),
    "Python code (out-of-domain)": (
        "def train_model(data, epochs=10):\n"
        "    optimizer = torch.optim.Adam(model.parameters(), lr=3e-4)\n"
        "    for epoch in range(epochs):\n"
        "        loss = compute_loss(model(data))\n"
        "        loss.backward()"
    ),
    "French text (out-of-domain)": (
        "Les réseaux de neurones artificiels sont des modèles de calcul inspirés "
        "du fonctionnement du cerveau humain. Ils sont utilisés dans de nombreux "
        "domaines comme la reconnaissance d'images et le traitement du langage naturel."
    ),
    "Social media (out-of-domain)": (
        "omg this new model is absolutely INSANE 🤯🔥🔥 can't believe how good it is "
        "lmaooo @elonmusk should see this https://t.co/abc123 #AI #NLP"
    ),
}

print(f"{'Domain':<35} {'Tokens':<10} {'Words':<10} {'Fertility'}")
print("─" * 70)
for domain, text in domain_samples.items():
    wiki_toks = wiki_tokenizer.encode(text).tokens
    words = text.split()
    fert = len(wiki_toks) / len(words) if words else 0
    print(f"{domain:<35} {len(wiki_toks):<10} {len(words):<10} {fert:.3f}")

print()
print("Higher fertility = more tokens per word = less efficient encoding.")
print("Out-of-domain text gets fragmented into more subtokens.")

> **Takeaway:** Always check your tokenizer's fertility on a sample from each data split and domain.
> A large gap between train and test fertility signals **domain mismatch** — the tokenizer is not well-suited to the target data and may need retraining or replacement.

### 5.6 — Common Tokenization Bugs Checklist

| # | Bug | Symptom | Fix |
|---|---|---|---|
| 1 | Forgetting `truncation=True` | Silent data loss or OOM on long sequences | Always set `truncation=True, max_length=...` |
| 2 | No `pad_token` for GPT-2 | `ValueError` when padding | `tokenizer.pad_token = tokenizer.eos_token` |
| 3 | Wrong tokenizer for model | Garbage output, wrong vocab size | Always load tokenizer from same checkpoint as model |
| 4 | Loss computed on pad tokens | Inflated/wrong loss, corrupted gradients | Mask labels with `-100` on pad positions |
| 5 | Right-padding for generation | Model "continues" from pad tokens | Use `padding_side="left"` for decoder models during generation |
| 6 | Assuming 1 word = 1 token | Wrong NER alignment, wrong positional assumptions | Use `word_ids()` or offset mappings |
| 7 | Silent truncation | No error, but data quietly lost | Set `truncation=True` explicitly and log when it fires |

In [ ]:
# --- Demo: Bug #2 — GPT-2 has no pad_token by default ---
buggy_tok = AutoTokenizer.from_pretrained("gpt2")

print("Bug #2: GPT-2 pad_token is:", repr(buggy_tok.pad_token))
try:
    buggy_tok(["Hello!", "How are you?"], padding=True, return_tensors="pt")
except Exception as e:
    print(f"Error: {type(e).__name__}: {e}")

# Fix:
buggy_tok.pad_token = buggy_tok.eos_token
result = buggy_tok(["Hello!", "How are you?"], padding=True, return_tensors="pt")
print(f"\nAfter fix (pad_token = eos_token): works! Shape = {result['input_ids'].shape}")

print("\n" + "─" * 60)

# --- Demo: Bug #6 — 1 word ≠ 1 token ---
word = "antidisestablishmentarianism"
tokens = bert_tok.tokenize(word)
print(f"\nBug #6: '{word}' → {len(tokens)} tokens: {tokens}")
print("Never assume len(words) == len(tokens)!")

### Exercise 5

1. Tokenize three IMDB reviews as a **causal LM batch** (pad left, mask labels with `-100`).
   Feed the batch to GPT-2 and print the loss. What happens if you forget to mask padding in labels?

2. Create a small NER example with at least one word that splits into **3+ subtokens**.
   Align the labels using `word_ids()`. Verify the label tensor has `-100` on continuation subtokens.

3. Compare fertility of our **wiki tokenizer** on:
   - A paragraph from Wikipedia
   - A paragraph of Python code
   - A paragraph of French text

   What does this tell you about tokenizer coverage?

In [ ]:
# Your answers here


## Going Further

- [HuggingFace Tokenizer docs](https://huggingface.co/docs/tokenizers)
- [The original BPE paper (Sennrich et al., 2016)](https://arxiv.org/abs/1508.07909)
- [Andrej Karpathy's minBPE](https://github.com/karpathy/minbpe) — a clean from-scratch implementation
- [SentencePiece](https://github.com/google/sentencepiece) — used by LLaMA, T5, mT5
- [Tiktoken](https://github.com/openai/tiktoken) — OpenAI's fast BPE tokenizer